# Алгоритмический baseline: проверка товарного знака по наименованию товара

**Критерий:** «Значение атрибута „Товарный знак“ не соответствует наименованию товара».

**Цель:** построить консервативный NLP/rule-based baseline, который возвращает `MATCH`, `LIKELY_MATCH`, `MISMATCH` или `REVIEW` и объясняет причину решения.

## Архитектура решения

1. Нормализация текста и mixed-script символов.
2. Проверка качества входа.
3. Обработка специальных значений «нет / отсутствует / без бренда».
4. Exact match.
5. Проверенные связи «аромат/линейка → бренд».
6. Статистические ассоциации из датасета.
7. Прямой конфликт другого бренда.
8. Fuzzy matching.
9. Trusted-brand conflict.
10. Если доказательств недостаточно — `REVIEW`.

In [1]:
from pathlib import Path
import re

import pandas as pd
from rapidfuzz import fuzz
from sklearn.metrics import precision_recall_fscore_support

In [2]:
DATA_PATH = Path("perfumery_brands.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Файл perfumery_brands.csv не найден. "
        "Положите его в ту же папку, что и ноутбук, и запустите ячейки заново."
    )

REQUIRED_COLUMNS = ["brand", "brand_in_text", "text"]

In [3]:
data = pd.read_csv(
    DATA_PATH,
    usecols=REQUIRED_COLUMNS
)

print("Размер датасета:", data.shape)
data.head()

Размер датасета: (1494163, 3)


,brand,text,brand_in_text
0,ЮДИФ,Туалетная вода <Юдиф> Вишневый цвет 42мл,ЮДИФ
1,ЮДИФ,Туалетная вода <Юдиф> Нежная сирень жен 42мл,ЮДИФ
2,SHAIK PLATINUM,Парфюмерная вода W 332 SHAIK Platınum Fruity 50ML,SHAIK
3,SHAIK PLATINUM,Парфюмерная вода MW 333 SHAIK Platınum Leather...,SHAIK
4,SHAIK PLATINUM,Парфюмерная вода W 382 SHAIK Platınum Floral G...,SHAIK


## Нормализация текста

Базовая нормализация приводит строки к нижнему регистру, заменяет `ё` на `е`, удаляет пунктуацию и лишние пробелы. Дополнительная нормализация исправляет смешение визуально похожих кириллических и латинских символов внутри одного слова.

In [4]:
def normalize_text(text):
    if pd.isna(text):
        return None

    text = str(text).lower()
    text = text.replace("ё", "е")
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [5]:
confusable_map = str.maketrans({
    "а": "a",
    "е": "e",
    "о": "o",
    "р": "p",
    "с": "c",
    "у": "y",
    "х": "x",
    "к": "k",
    "м": "m",
    "т": "t",
    "в": "b",
    "н": "h",
})


def normalize_for_matching(text):
    text = normalize_text(text)

    if text is None:
        return None

    result = []

    for word in text.split():

        has_latin = bool(
            re.search(r"[a-z]", word)
        )

        has_cyrillic = bool(
            re.search(r"[а-яё]", word)
        )

        # Исправляем только слова,
        # где одновременно есть латиница и кириллица
        if has_latin and has_cyrillic:
            word = word.translate(confusable_map)

        result.append(word)

    return " ".join(result)

## Служебные правила и проверенные связи

В этом блоке фиксируются специальные значения отсутствующего бренда, небольшое число подтверждённых алиасов и вручную проверенные связи «название аромата/линейки → бренд».

In [6]:
no_brand_values = {
    "без товарного знака",
    "безтоварного знака",
    "без бренда",
    "бренд отсутствует",
    "товарный знак отсутствует",
    "отсутствует",
    "нет",
    "без знака",
    "без товарной марки",
    "не указан",
}

brand_aliases = {
    "byredo parfums": "byredo",
    "francis kurkdjian": "maison francis kurkdjian",
    "parfum francis kurkdjian": "maison francis kurkdjian",
}

verified_product_relations = {
    "domenico gusto": "christine lavoisier parfums",
    "oudi": "al haramain",
    "russian police clean hands": "sergio nero",
    "the only one": "dolce gabbana",
    "stronger with you intensely": "giorgio armani",
    "festival splashes": "mexx",
    "ma cherry": "adopt",
    "hyrax": "zoologist",
}

## Построение статистических справочников

In [7]:
brand_data = data[["brand", "brand_in_text"]].copy()

brand_data["brand_normalized"] = brand_data["brand"].apply(normalize_text)
brand_data["brand_in_text_normalized"] = brand_data["brand_in_text"].apply(normalize_text)

clean_pairs_data = brand_data.dropna(
    subset=["brand_normalized", "brand_in_text_normalized"]
).copy()

clean_pairs_data = clean_pairs_data[
    (~clean_pairs_data["brand_normalized"].isin(no_brand_values))
    & (~clean_pairs_data["brand_normalized"].str.isnumeric())
]

clean_pair_counts = (
    clean_pairs_data
    .groupby(["brand_in_text_normalized", "brand_normalized"])
    .size()
    .reset_index(name="count")
)

# Доверенный справочник прямых брендов
clean_dominant_brand = (
    clean_pair_counts
    .sort_values(["brand_in_text_normalized", "count"], ascending=[True, False])
    .drop_duplicates("brand_in_text_normalized")
    .rename(columns={"brand_normalized": "dominant_brand", "count": "dominant_count"})
)

clean_candidate_totals = (
    clean_pair_counts
    .groupby("brand_in_text_normalized")["count"]
    .sum()
    .reset_index(name="total")
)

clean_dominant_stats = clean_dominant_brand.merge(
    clean_candidate_totals,
    on="brand_in_text_normalized"
)
clean_dominant_stats["dominance"] = (
    clean_dominant_stats["dominant_count"] / clean_dominant_stats["total"]
)

trusted_brands = clean_dominant_stats[
    (clean_dominant_stats["total"] >= 20)
    & (clean_dominant_stats["dominance"] >= 0.95)
    & (
        clean_dominant_stats["brand_in_text_normalized"]
        == clean_dominant_stats["dominant_brand"]
    )
].copy()

trusted_brand_catalog = trusted_brands["brand_in_text_normalized"].tolist()

print("Доверенных брендов:", len(trusted_brand_catalog))

Доверенных брендов: 316


In [8]:
def find_trusted_brands(text):
    text = normalize_text(text)

    if text is None:
        return []

    text_with_spaces = f" {text} "
    found = []

    for brand in trusted_brand_catalog:
        if len(brand) < 4:
            continue

        brand_with_spaces = f" {brand} "

        if brand_with_spaces in text_with_spaces:
            found.append(brand)

    return sorted(found, key=len, reverse=True)

In [9]:
def brands_equivalent(brand_1, brand_2):
    brand_1 = normalize_text(brand_1)
    brand_2 = normalize_text(brand_2)

    if brand_1 is None or brand_2 is None:
        return False

    # Полное совпадение
    if brand_1 == brand_2:
        return True

    # Один вариант входит в другой
    if brand_1 in brand_2 or brand_2 in brand_1:
        return True

    # Небольшая разница в написании
    similarity = fuzz.ratio(brand_1, brand_2)

    if similarity >= 90:
        return True

    return False

In [10]:
def detect_trusted_mismatch(brand, text):
    detected_brands = find_trusted_brands(text)

    conflicts = []

    for detected_brand in detected_brands:
        if not brands_equivalent(brand, detected_brand):
            conflicts.append(detected_brand)

    if conflicts:
        return "MISMATCH", conflicts

    return "NO_CONFLICT_FOUND", []

In [11]:
def canonicalize_brand(brand):
    brand = normalize_text(brand)

    if brand is None:
        return None

    return brand_aliases.get(brand, brand)

In [12]:
canonical_pairs = clean_pair_counts.copy()
canonical_pairs["canonical_brand"] = canonical_pairs["brand_normalized"].apply(canonicalize_brand)

canonical_pair_counts = (
    canonical_pairs
    .groupby(["brand_in_text_normalized", "canonical_brand"], as_index=False)["count"]
    .sum()
)

canonical_ranked = canonical_pair_counts.copy()
canonical_ranked["rank"] = (
    canonical_ranked
    .groupby("brand_in_text_normalized")["count"]
    .rank(method="first", ascending=False)
)

canonical_top1 = (
    canonical_ranked[canonical_ranked["rank"] == 1]
    [["brand_in_text_normalized", "canonical_brand", "count"]]
    .rename(columns={"canonical_brand": "top1_brand", "count": "top1_count"})
)

canonical_top2 = (
    canonical_ranked[canonical_ranked["rank"] == 2]
    [["brand_in_text_normalized", "canonical_brand", "count"]]
    .rename(columns={"canonical_brand": "top2_brand", "count": "top2_count"})
)

canonical_strength = canonical_top1.merge(
    canonical_top2,
    on="brand_in_text_normalized",
    how="left"
)

canonical_strength["top_ratio"] = (
    canonical_strength["top1_count"] / canonical_strength["top2_count"]
)
canonical_strength["count_gap"] = (
    canonical_strength["top1_count"] - canonical_strength["top2_count"]
)

strong_associations = canonical_strength[
    (canonical_strength["top1_count"] >= 20)
    & (canonical_strength["top_ratio"] >= 3)
    & (canonical_strength["count_gap"] >= 20)
    & (canonical_strength["brand_in_text_normalized"].str.len() >= 4)
].copy()

association_map = dict(
    zip(
        strong_associations["brand_in_text_normalized"],
        strong_associations["top1_brand"]
    )
)

print("Устойчивых статистических связей:", len(association_map))

Устойчивых статистических связей: 747


In [13]:
def find_associated_brands(text):
    normalized_text = normalize_for_matching(text)

    if normalized_text is None:
        return []

    matches = []

    for term, associated_brand in association_map.items():
        normalized_term = normalize_for_matching(term)

        if normalized_term is None:
            continue

        # Ищем отдельное слово или фразу,
        # а не часть другого слова
        pattern = rf"(?<!\w){re.escape(normalized_term)}(?!\w)"

        for match in re.finditer(pattern, normalized_text):
            matches.append(
                {
                    "term": term,
                    "brand": associated_brand,
                    "start": match.start(),
                    "end": match.end(),
                    "length": match.end() - match.start()
                }
            )

    # Сначала самые длинные и конкретные совпадения
    matches.sort(
        key=lambda item: item["length"],
        reverse=True
    )

    selected = []

    for candidate in matches:
        # Проверяем, не находится ли кандидат
        # полностью внутри уже выбранного более длинного совпадения
        is_nested = any(
            candidate["start"] >= existing["start"]
            and candidate["end"] <= existing["end"]
            for existing in selected
        )

        if not is_nested:
            selected.append(candidate)

    # Технические координаты наружу не возвращаем
    return [
        {
            "term": item["term"],
            "brand": item["brand"]
        }
        for item in selected
    ]

## Дополнительные проверки

In [14]:
def find_verified_product_relation(text):
    normalized_text = normalize_for_matching(text)

    if normalized_text is None:
        return None

    # Сначала проверяем более длинные названия
    for term in sorted(
        verified_product_relations,
        key=len,
        reverse=True
    ):
        normalized_term = normalize_for_matching(term)

        # Ищем термин только как отдельное слово или фразу,
        # а не как часть другого слова
        pattern = rf"(?<!\w){re.escape(normalized_term)}(?!\w)"

        if re.search(pattern, normalized_text):
            return {
                "term": term,
                "brand": verified_product_relations[term]
            }

    return None

In [15]:
def check_input_quality(text):
    if pd.isna(text):
        return {
            "is_valid": False,
            "reason": "Отсутствует текст товара"
        }

    text = str(text)

    text_length = len(text)

    barcode_like_count = len(
        re.findall(
            r"\b\d{12,14}\b",
            text
        )
    )

    if text_length > 500:
        return {
            "is_valid": False,
            "text_length": text_length,
            "barcode_like_count": barcode_like_count,
            "reason": "Аномально длинное наименование товара"
        }

    if barcode_like_count >= 3:
        return {
            "is_valid": False,
            "text_length": text_length,
            "barcode_like_count": barcode_like_count,
            "reason": (
                "В тексте обнаружено несколько числовых "
                "идентификаторов, возможна склейка товаров"
            )
        }

    return {
        "is_valid": True,
        "text_length": text_length,
        "barcode_like_count": barcode_like_count
    }

In [16]:
def check_brand_input_quality(brand):
    if pd.isna(brand):
        return {
            "is_valid": False,
            "reason": "Отсутствует значение товарного знака"
        }

    normalized_brand = normalize_for_matching(brand)

    if normalized_brand is None:
        return {
            "is_valid": False,
            "reason": "Отсутствует значение товарного знака"
        }

    # 1. Значение состоит только из цифр
    if normalized_brand.isdigit():
        return {
            "is_valid": False,
            "reason": (
                "Значение товарного знака состоит только из цифр "
                "и требует отдельной проверки корректности атрибута"
            )
        }

    # 2. Сомнительные служебные значения текущего датасета
    suspicious_brand_values = {
        "aroma"
    }

    if normalized_brand in suspicious_brand_values:
        return {
            "is_valid": False,
            "reason": (
                "Значение товарного знака является сомнительным "
                "служебным значением в текущем датасете"
            )
        }

    # 3. Значение похоже на наименование юридического лица
    legal_entity_patterns = [
        r"^(ооо|оао|зао|пао|ао)\b",
        r"^ип\b",
        r"^общество с ограниченной ответственностью\b",
        r"^(llc|ltd|inc)\b"
    ]

    for pattern in legal_entity_patterns:
        if re.search(pattern, normalized_brand):
            return {
                "is_valid": False,
                "reason": (
                    "Значение похоже на наименование юридического "
                    "лица или производителя и требует отдельной проверки"
                )
            }

    return {
        "is_valid": True
    }

In [17]:
def find_direct_brand_conflict(brand, text):
    declared_brand = normalize_for_matching(
        canonicalize_brand(brand)
    )

    associations = find_associated_brands(text)

    for association in associations:
        term = normalize_for_matching(
            association["term"]
        )

        associated_brand = normalize_for_matching(
            canonicalize_brand(
                association["brand"]
            )
        )

        # Сильный случай:
        # найденный термин сам является названием бренда
        if term == associated_brand:

            # Если найденный бренд отличается от заявленного
            if declared_brand != associated_brand:
                return {
                    "brand": associated_brand,
                    "term": association["term"]
                }

    return None

## Финальная inference-функция

In [18]:
def check_trademark(brand, text):
    # Нормализация
    normalized_brand = normalize_for_matching(brand)
    normalized_text = normalize_for_matching(text)

    # 1. Проверка отсутствующих данных
    if normalized_brand is None or normalized_text is None:
        return {
            "status": "REVIEW",
            "declared_brand": brand,
            "method": "missing_data",
            "reason": "Отсутствуют необходимые данные"
        }

    # 2. Проверка качества входного текста
    quality_result = check_input_quality(text)

    if not quality_result["is_valid"]:
        return {
            "status": "REVIEW",
            "declared_brand": brand,
            "method": "input_quality_review",
            "text_length": quality_result.get("text_length"),
            "barcode_like_count": quality_result.get(
                "barcode_like_count"
            ),
            "reason": quality_result["reason"]
        }

    # 3. Значения:
    # "Без товарного знака", "нет", "отсутствует", "не указан"
    if normalized_brand in no_brand_values:
        detected_brands = find_trusted_brands(text)

        if detected_brands:
            return {
                "status": "REVIEW",
                "declared_brand": brand,
                "detected_brands": detected_brands,
                "method": "no_brand_conflict_review",
                "reason": (
                    "Указано отсутствие товарного знака, "
                    "но в наименовании найдено упоминание бренда"
                )
            }

        return {
            "status": "REVIEW",
            "declared_brand": brand,
            "method": "no_brand_review",
            "reason": (
                "Указано отсутствие товарного знака. "
                "Недостаточно данных для автоматического подтверждения"
            )
        }

    # 4. Проверка формата значения товарного знака
    brand_quality_result = check_brand_input_quality(brand)

    if not brand_quality_result["is_valid"]:
        return {
            "status": "REVIEW",
            "declared_brand": brand,
            "method": "invalid_brand_format_review",
            "reason": brand_quality_result["reason"]
        }

    # 5. Точное вхождение бренда в наименование
    if normalized_brand in normalized_text:
        return {
            "status": "MATCH",
            "declared_brand": brand,
            "similarity": 100,
            "method": "exact_match",
            "reason": "Товарный знак найден в наименовании товара"
        }

    # 6. Проверенные вручную связи:
    # название аромата / линейки -> реальный бренд
    verified_relation = find_verified_product_relation(text)

    if verified_relation is not None:
        declared_canonical = normalize_for_matching(
            canonicalize_brand(brand)
        )

        verified_canonical = normalize_for_matching(
            canonicalize_brand(
                verified_relation["brand"]
            )
        )

        # Связь подтверждает заявленный бренд
        if declared_canonical == verified_canonical:
            return {
                "status": "MATCH",
                "declared_brand": brand,
                "associated_term": verified_relation["term"],
                "associated_brand": verified_relation["brand"],
                "method": "verified_relation_match",
                "reason": (
                    "В наименовании найдено название аромата "
                    "или линейки, связанное с заявленным товарным знаком"
                )
            }

        # Связь указывает на другой бренд
        return {
            "status": "MISMATCH",
            "declared_brand": brand,
            "detected_brands": [
                verified_relation["brand"]
            ],
            "associated_term": verified_relation["term"],
            "method": "verified_relation_conflict",
            "reason": (
                "В наименовании найдено название аромата "
                "или линейки, связанное с другим товарным знаком"
            )
        }

    # 7. Автоматически найденные устойчивые ассоциации
    associations = find_associated_brands(text)

    declared_canonical = normalize_for_matching(
        canonicalize_brand(brand)
    )

    for association in associations:
        associated_canonical = normalize_for_matching(
            canonicalize_brand(
                association["brand"]
            )
        )

        # Ассоциация подтверждает заявленный бренд
        if declared_canonical == associated_canonical:
            return {
                "status": "MATCH",
                "declared_brand": brand,
                "associated_term": association["term"],
                "associated_brand": association["brand"],
                "method": "association_match",
                "reason": (
                    "В наименовании найдено название, "
                    "устойчиво связанное с заявленным товарным знаком"
                )
            }

    # 8. Проверка прямого упоминания другого бренда
    direct_conflict = find_direct_brand_conflict(
        brand,
        text
    )

    if direct_conflict is not None:
        return {
            "status": "MISMATCH",
            "declared_brand": brand,
            "detected_brands": [
                direct_conflict["brand"]
            ],
            "associated_term": direct_conflict["term"],
            "method": "direct_brand_conflict",
            "reason": (
                "В наименовании явно обнаружен другой "
                "товарный знак"
            )
        }

    # 9. Fuzzy matching
    similarity = fuzz.partial_ratio(
        normalized_brand,
        normalized_text
    )

    if similarity >= 90:
        return {
            "status": "LIKELY_MATCH",
            "declared_brand": brand,
            "similarity": similarity,
            "method": "fuzzy_match",
            "reason": (
                "Обнаружено высокое сходство "
                "товарного знака и наименования"
            )
        }

    # 10. Поиск другого доверенного бренда
    conflict_status, conflicts = detect_trusted_mismatch(
        brand,
        text
    )

    if conflict_status == "MISMATCH":
        return {
            "status": "MISMATCH",
            "declared_brand": brand,
            "detected_brands": conflicts,
            "similarity": similarity,
            "method": "trusted_brand_conflict",
            "reason": (
                "В наименовании обнаружен другой "
                "доверенный товарный знак"
            )
        }

    # 11. Недостаточно информации
    return {
        "status": "REVIEW",
        "declared_brand": brand,
        "similarity": similarity,
        "method": "insufficient_evidence",
        "reason": (
            "Недостаточно данных для автоматического решения"
        )
    }

In [19]:
tests = [
    ("PARFUMS DE MARLY", "Parfums de Marly Delina Exclusif"),
    ("Byredo", "Парфюмерная вода Gypsy Water"),
    ("Franck Oliver", "FRANCK OLIVIER IN PINK"),
    ("Dior", "Police The Legendary Scent"),
    ("НЕ УКАЗАН", "PRADA AMBER POUR HOMME"),
    ("Mr.Tooth", "M.MICALLEF парфюмерная вода ДЕСИРТОКСИК 100ml"),
]

for brand, text in tests:
    result = check_trademark(brand, text)
    print(f"{brand!r} -> {result['status']} / {result.get('method')}")

'PARFUMS DE MARLY' -> MATCH / exact_match
'Byredo' -> MATCH / association_match
'Franck Oliver' -> MISMATCH / direct_brand_conflict
'Dior' -> MISMATCH / direct_brand_conflict
'НЕ УКАЗАН' -> REVIEW / no_brand_conflict_review
'Mr.Tooth' -> MISMATCH / direct_brand_conflict


## Проверка на случайной выборке 5000

Для воспроизводимости используется `random_state=42`. Последний подтверждённый прогон финальной версии: **2636 MATCH / 2251 REVIEW / 93 MISMATCH / 20 LIKELY_MATCH**.

In [ ]:
random_df = data[["brand", "text", "brand_in_text"]].sample(
    n=5000,
    random_state=42
).copy()

random_df["check_result"] = random_df.apply(
    lambda row: check_trademark(row["brand"], row["text"]),
    axis=1
)
random_df["final_status"] = random_df["check_result"].apply(lambda x: x["status"])
random_df["final_method"] = random_df["check_result"].apply(lambda x: x.get("method"))

print(random_df["final_status"].value_counts())
print()
print(random_df["final_method"].value_counts())

## Итоговая ручная оценка

Ручная контрольная выборка: **100 товаров**. Метки: **68 MATCH, 19 MISMATCH, 13 UNCERTAIN**. `UNCERTAIN` исключены из основных бинарных метрик. `LIKELY_MATCH` при оценке считается направленным решением MATCH; `REVIEW` считается отказом от автоматического решения.

In [ ]:
# Итоговая ручная оценка: 100 товаров, из них 13 UNCERTAIN.
manual_label_distribution = pd.Series({
    "MATCH": 68,
    "MISMATCH": 19,
    "UNCERTAIN": 13,
}, name="count")

final_confusion = pd.DataFrame(
    {
        "MATCH": [50, 0],
        "MISMATCH": [1, 3],
        "REVIEW": [17, 16],
    },
    index=pd.Index(["MATCH", "MISMATCH"], name="manual_label")
)
final_confusion.columns.name = "predicted_label"

# Восстанавливаем y_true/y_pred из подтверждённой матрицы, чтобы пересчитать метрики.
y_true = (
    ["MATCH"] * 68
    + ["MISMATCH"] * 19
)
y_pred = (
    ["MATCH"] * 50
    + ["MISMATCH"] * 1
    + ["REVIEW"] * 17
    + ["MISMATCH"] * 3
    + ["REVIEW"] * 16
)

precision, recall, f1, support = precision_recall_fscore_support(
    y_true,
    y_pred,
    labels=["MATCH", "MISMATCH"],
    zero_division=0
)

metrics_df = pd.DataFrame({
    "class": ["MATCH", "MISMATCH"],
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "support": support,
})

automatic_total = 50 + 1 + 3
automatic_correct = 50 + 3
coverage = automatic_total / 87
automatic_accuracy = automatic_correct / automatic_total

print("Распределение ручных меток:")
print(manual_label_distribution)
print("\nМатрица итоговой оценки:")
display(final_confusion)
print(f"Coverage: {coverage:.2%}")
print(f"Accuracy среди автоматических решений: {automatic_accuracy:.2%}")
display(metrics_df)

## Итог

- **Coverage:** 62.07%
- **Accuracy среди автоматических решений:** 98.15%
- **MATCH:** precision 100%, recall 73.53%, F1 84.75%
- **MISMATCH:** precision 75%, recall 15.79%, F1 26.09%

Baseline хорошо работает, когда принимает уверенное решение, но пропускает большую часть реальных MISMATCH в `REVIEW`. Это обосновывает гибридную архитектуру: правила остаются первым уровнем, а ML обрабатывает сложные REVIEW-кейсы.

## Ограничения и методологические замечания

- `association_map` в текущем прототипе построен по общему датасету; для строгой итоговой оценки его нужно строить только на train после split, чтобы исключить leakage.
- Пороговые значения (`0.95`, `20`, `3`, `20`, fuzzy `90`) являются baseline-настройками и требуют отдельной валидации.
- Список verified relations небольшой и вручную подтверждённый; его нужно хранить как отдельный управляемый справочник.
- `REVIEW` — намеренный безопасный исход, а не ошибка: система воздерживается от автоматического решения при слабых доказательствах.

## Следующий этап — ML для REVIEW

Рекомендуемый старт: сформировать честный train/test split, подготовить размеченные пары `brand + text → MATCH/MISMATCH`, начать с TF‑IDF + Logistic Regression и сравнить recall/precision с baseline. ML не должен автоматически заменять правила: его цель — уменьшить долю REVIEW и поднять recall MISMATCH без заметной потери precision.

## Справочные материалы

- RapidFuzz `partial_ratio`: https://rapidfuzz.github.io/RapidFuzz/Usage/fuzz.html
- scikit-learn precision/recall: https://scikit-learn.org/stable/auto_examples/model_selection/plot_precision_recall.html
- scikit-learn `classification_report`: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html